# OMBRIA sensor-state v0.2 Full follow-up
Run only after the v0.2 smoke archive passes review. Enable a GPU and Internet, then choose **Run All**. The five-repeat, seven-route run is expected to take approximately 4–6 hours on P100.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

WORKING = Path('/kaggle/working')
REPO_URL = 'https://github.com/NewRudy/geoai-ombria-robustness.git'
TAG = 'v0.2.0-sensor-state'
project = WORKING / 'geoai-ombria-robustness'
os.chdir(WORKING)
if project.exists():
    shutil.rmtree(project)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', TAG, REPO_URL, str(project)], check=True)
os.chdir(project)
print('locked source:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24', 'pillow>=10.0', 'matplotlib>=3.7'], check=True)
subprocess.run([sys.executable, 'scripts/ensure_cuda_compat.py'], check=True)
subprocess.run([sys.executable, 'scripts/check_cuda_runtime.py'], check=True)
scripts = [str(path) for path in Path('scripts').glob('*.py')]
subprocess.run([sys.executable, '-m', 'py_compile', *scripts], check=True)

In [ ]:
env = os.environ.copy()
env.update({'MODE': 'full', 'EPOCHS': '25', 'BATCH_SIZE': '8', 'BASE_CHANNELS': '16', 'PYTHON': sys.executable})
subprocess.run(['bash', 'scripts/run_sensor_state_v2_matrix.sh'], check=True, env=env)

In [ ]:
import hashlib, json
from zipfile import ZipFile
from IPython.display import FileLink, display

artifact = project / 'results' / 'ombria_sensor_state_v2_artifacts.zip'
with ZipFile(artifact) as archive:
    assert archive.testzip() is None
    names = archive.namelist()
    artifact_manifest_name = next(name for name in names if name.endswith('artifact_manifest.json'))
    artifact_manifest = json.loads(archive.read(artifact_manifest_name))
    for record in artifact_manifest['files']:
        assert record['path'] in names
        assert hashlib.sha256(archive.read(record['path'])).hexdigest() == record['sha256']
    checkpoint_manifest_name = next(name for name in names if name.endswith('checkpoint_manifest.json'))
    checkpoint_manifest = json.loads(archive.read(checkpoint_manifest_name))
    assert checkpoint_manifest['weights_included'] is True
    assert len(checkpoint_manifest['checkpoints']) == 70
    assert len([name for name in names if name.endswith('best_clean.pt')]) == 35
    assert len([name for name in names if name.endswith('best_robust.pt')]) == 35
print(f'validated: {artifact} ({artifact.stat().st_size / 1024**2:.1f} MiB)')
display(FileLink(str(artifact)))